# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c23_design_space_optimizer     
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-19
### Design space optimization script
**Goal:** Optimize the structure by iterating over structural performance, circular performance, and total cost.
**Inputs:**
*   Search space definition
*   Trained surrogate model
*   Material input dataset

**Outputs:**
*   Optimized design candidates
*   Ranked solution set

# INPUT

In [ ]:
import importlib

from workflows import c23_stage_geometry as stage_geometry
from workflows import c23_stage_optimizer as stage_optimizer

importlib.reload(stage_geometry)
importlib.reload(stage_optimizer)

SEED = 42

testing = False
optimizer_context = stage_optimizer.prepare_optimizer_context(testing=testing, seed=SEED)
optimizer_search_space = optimizer_context["optimizer_search_space"]
df_input_stock = optimizer_context["df_input_stock"]
SURROGATE_BUNDLE = optimizer_context["SURROGATE_BUNDLE"]
SURROGATE_BUNDLE_ERROR = optimizer_context["SURROGATE_BUNDLE_ERROR"]
GA_CONFIG = optimizer_context["GA_CONFIG"]
FIXED_NORMALIZATION_CONSTANTS = optimizer_context["FIXED_NORMALIZATION_CONSTANTS"]
BOUNDS_SOURCE_INFO = optimizer_context["BOUNDS_SOURCE_INFO"]
MODEL_PREFIX = optimizer_context["MODEL_PREFIX"]
MODEL_PREFIX_COMPLEX = optimizer_context["MODEL_PREFIX_COMPLEX"]
MODEL_PREFIX_SIMPLE = optimizer_context["MODEL_PREFIX_SIMPLE"]
SURROGATE_EDGE_FEATURE_MODE = optimizer_context["SURROGATE_EDGE_FEATURE_MODE"]

print(f"Search space variables: {len(optimizer_search_space)}")
print(f"Stock elements: {len(df_input_stock)}")
print(f"Fixed normalization constants (initial): {FIXED_NORMALIZATION_CONSTANTS}")

# OPTIMIZER

## GA Strategy & Search Space

This notebook implements a **custom Genetic Algorithm (GA)** to optimize timber structures by iterating over the entire design workflow (c24-c27).

### What the GA is Doing

**Goal:** Minimize embodied carbon while maximizing reuse of reclaimed timber and maintaining structural adequacy.

**Search Strategy:**
1. **Initialization**: Generate 30 random candidate designs by sampling 18 design variables from the search space
2. **Evaluation**: For each design, run the full pipeline:
   - Generate geometry (vertices/edges) from design parameters
   - Predict member forces using GNN surrogate model
   - Compute utilization matrix for all members
   - Build cost matrix (carbon footprint for each slot-stock pairing)
   - Solve MILP optimization to find optimal timber assignment
   - Calculate multi-objective fitness (cost + reuse rate + structural check)
3. **Selection & Reproduction**: Retain best designs (elitism), breed new candidates through:
   - **Tournament selection**: Pick 3 random individuals, keep fittest
   - **Crossover**: Blend two parent designs (50/50 mix for continuous vars, random choice for discrete)
   - **Mutation**: Randomly perturb ~25% of parameters in each child design
4. **Iteration**: Repeat evaluation + selection for 30 generations

**Convergence**: By generation 30, the algorithm converges to high-fitness designs (low cost, high reuse).

### How the GA Moves Points

The **18 design variables** control the geometry by modifying node positions and proportions:

| Variable Type | Count | Controls | Movement Mechanism |
|---|---|---|---|
| **Vertex shifts** (X, Y, Z) | 9 | Position of nodes | Continuous mutation (Gaussian perturbation) + crossover (linear blend) |
| **Bilinear parameters** (U, V) | 6 | Proportional grid scaling | Continuous mutation (scaled by search space span) + crossover |
| **Discrete parameters** | 3 | Topology/feature selection | Random swap between categorical options (e.g., span type) |

**Mutation Details:**
- Each parameter has ~25% chance to mutate per generation
- **Continuous variables** (vertex shifts, bilinear params): Add Gaussian noise (σ = 10% of search range), then clip to bounds
  - Example: If Y-shift range is [−0.5m, +0.5m], mutation adds noise up to ±0.05m
  - This gradually explores neighborhood around current solution
- **Discrete variables**: Randomly swap to alternative option (e.g., span_type: "simple" → "complex")

**Crossover Details:**
- Each parent contributes to child with 50% probability (arithmetic mean for continuous, random pick for discrete)
- Example: If Parent A has Y-shift=−0.3m and Parent B has Y-shift=+0.2m, child could be Y-shift=−0.05m
- This explores design space between successful parents

**Result**: Population gradually shifts toward low-cost, high-reuse designs. Node positions are refined through many small mutations; radically different topologies emerge through crossover of diverse parents.


In [ ]:
print(f"GA operators are available from workflows.c23_stage_optimizer.")
print(f"Population size (default): {GA_CONFIG['population_size']}")

### GA Operators: Selection, Crossover & Mutation

**Tournament Selection**
- Randomly picks `k=3` individuals from population
- Selects the one with best (lowest) fitness
- Biased toward better solutions but preserves diversity (weaker candidates still have chance if picked)

**Crossover (Point Blending)**
- Takes two parent designs (parameter dicts)
- For each of the 18 design variables:
  - **Continuous** (vertex shifts, bilinear params): Child value = α × Parent_A + (1−α) × Parent_B, where α ∈ [0,1] random
  - **Discrete** (categorical options): Child randomly chooses Parent_A's or Parent_B's option
- Result: New candidate that inherits traits from both parents (explores design space between them)

**Mutation (Parameter Perturbation)**
- For each of 18 design variables, ~25% chance to mutate:
  - **Continuous**: Add Gaussian noise scaled to 10% of search range, then clip to bounds
    - Allows fine-tuning within bounds; prevents extreme jumps
  - **Discrete**: Swap to random alternative option from list
- Result: Introduces variation; prevents population from converging to local optimum

**Elitism**
- Keep top 2 designs from previous generation unchanged in next population
- Ensures best solutions never disappear
- Accelerates convergence to high-fitness region


### Why Points Move in Certain Directions: The Fitness Feedback Loop

**The key insight:** The GA doesn't have explicit rules like "move vertex X higher" or "increase span length." Instead, direction emerges from **fitness-driven selection**.

**How it works:**

1. **Mutation is Blind (Random Direction)**
   - Each mutation adds Gaussian noise with equal probability in all directions
   - Parent design has Y_shift = +0.2m → mutation might move it to +0.3m (up) or +0.1m (down) with ~equal probability
   - Discrete variables randomly swap between options with no bias

2. **Selection Filters by Fitness (Creates Direction)**
   - After all 30 individuals are evaluated, compare their fitness scores
   - **Best designs reproduce** (kept in population via elitism, selected for crossover)
   - **Worst designs die out** (discarded when selecting tournament winners)
   - This is the crucial feedback: if designs with higher Y_shift tend to produce lower embodied carbon, they survive; if lower Y_shift produces better fitness, those survive instead

3. **Crossover Exploits Discovered Patterns (Reinforces Direction)**
   - When two high-fitness parents both have Y_shift ≈ +0.25m, crossover tends to keep that range
   - Their offspring inherits Y_shift near +0.25m, with small mutation perturbations
   - Over generations, the population converges around parameter values that **correlate with low fitness**

4. **Result: Emergent Direction**
   - No central planner decides "move Y-shift higher"
   - But if geometry designs with higher Y-shift happen to reduce member utilization → lower cost → better fitness → those designs get selected
   - Population gradually drifts upward in Y-shift across generations
   - Different parameters drift different directions based on their correlation with cost, reuse rate, waste volume

**Example Trajectory:**
```
Generation 1: Population Y_shift range = [−0.5, +0.5] (random)
              → Evaluate all 30 designs
              → Observe: designs with Y_shift > 0 have better fitness (lower carbon)

Generation 2: Tournament selection favors Y_shift > 0 designs
              → Best designs reproduce
              → New population Y_shift range ≈ [−0.2, +0.4]
              → Mutation adds noise (still random direction)
              → But population mean drifts upward

Generation 30: Convergence
              → Best designs all cluster around Y_shift ≈ +0.3 to +0.35m
              → Small mutations explore ±0.05m neighborhood
              → But rarely drift back downward (selection pressure)
```

**Key Takeaway:** The GA is an **adaptive random search** guided by fitness. Mutation explores randomly, but selection acts like a compass pointing toward low-carbon, high-reuse designs. Emergent movement directions reflect the underlying **design problem landscape**—which parameter combinations actually lead to good structures.


## 24_ Geometry

In [ ]:
# Optional one-sample geometry sanity check before GA run
geometry_out = stage_geometry.run_random_geometry_stage(
    optimizer_search_space=optimizer_search_space,
    sample_id=0,
)

df_vertices_example = geometry_out["df_vertices"]
df_edges_example = geometry_out["df_edges"]
df_geometry_overview_example = geometry_out["df_geometry_overview"]

print(
    f"Example geometry: {len(df_vertices_example)} nodes, "
    f"{len(df_edges_example)} members"
 )
display(df_geometry_overview_example[["edge_id", "V1", "V2", "length_m"]].head(5))

## 24-27 Iteration

### GA Evaluation Pipeline: How Designs Are Scored

Each design candidate in the population is evaluated through a 5-stage pipeline to compute its fitness:

| Stage | Input | Output | Purpose |
|---|---|---|---|
| **c25 Feasibility** | Geometry + stock data | Matrix n x m with inf for infeasible combinations | Use GNN model to predict forces; check if members can handle loads |
| **c26 Cost Matrix** | Utilization matrix + stock catalog | Carbon cost for each (slot, stock) pair | Build optimization problem: which reclaimed/new timber for each member? |
| **c27 MILP** | Cost matrix | Optimal assignment + total cost | Solve integer program: minimize embodied carbon via timber selection |
| **c24 Fitness** | MILP results + design metrics | Normalized multi-objective score | Combine cost, reuse %, waste volume into single fitness value |

**Fitness Calculation:**
```
fitness = weighted_sum(
    cost_normalized = total_carbon / 8.0,           # Normalize to reference 8 kg CO2e
    reuse_normalized = (100 - reuse_rate) / 100,   # Prefer high reuse (normalize as 0-1)
    waste_normalized = waste_volume / 0.4            # Normalize to reference 0.4 m³
)
```
*(Weights depend on `weight_strategy`; default is "cost-dominant")*

**Failure Handling:**
- If MILP solver fails (infeasible timber assignment): Assign penalty fitness = 1,000,000 (very bad)
- If geometry generation or structural check errors: Penalty fitness applied, GA continues
- Low-fitness individuals are less likely to be selected for reproduction → pressure toward feasible, efficient designs

**Result:** After 30 generations, the best design in population is returned with its geometry, timber assignment, and performance metrics.


In [ ]:
print("One-time normalization constants ready:", FIXED_NORMALIZATION_CONSTANTS)
print("Normalization source:", BOUNDS_SOURCE_INFO)

print("Single-candidate evaluator ready.")
test_design, test_eval = stage_optimizer.run_evaluator_smoke_test(
    optimizer_search_space=optimizer_search_space,
    df_input_stock=df_input_stock,
    bundle=SURROGATE_BUNDLE,
    fixed_norm_constants=FIXED_NORMALIZATION_CONSTANTS,
    ga_config=GA_CONFIG,
)
print(f"Evaluator smoke test status: {test_eval['status']}")
if test_eval["reason"] is not None:
    print(f"Reason: {test_eval['reason']}")

# OUTPUT

In [ ]:
ga_result = stage_optimizer.run_ga_optimization(
    optimizer_search_space=optimizer_search_space,
    df_input_stock=df_input_stock,
    bundle=SURROGATE_BUNDLE,
    fixed_norm_constants=FIXED_NORMALIZATION_CONSTANTS,
    ga_config=GA_CONFIG,
)

ga_history_df = ga_result["ga_history_df"]
ga_ranked_df = ga_result["ga_ranked_df"]
best_overall = ga_result["best_overall"]

## GA Analysis & Diagnostics

**Purpose:** Deep inspection of GA behavior to catch anomalies, validate convergence, and understand design patterns.

**Included Analyses:**
- **Convergence**: Best/mean/worst fitness per generation (detects stalling or divergence)
- **Feasibility**: % viable solutions per generation (detects infeasibility waves)
- **Parameter Correlation**: Which design variables correlate with low fitness?
- **Elite Performance**: How well did selected individuals perform?
- **Fitness Distribution**: Histogram to spot bimodal/weird distributions
- **Top Designs**: Ranked table with all key metrics
- **Anomaly Detection**: Flags unusual fitness spikes, all-infeasible generations, parameter outliers
- **Text Summary**: Key findings + warnings about potential issues


In [ ]:
analysis_result = stage_optimizer.run_ga_analysis(
    ga_history_df=ga_history_df,
    ga_ranked_df=ga_ranked_df,
    optimizer_search_space=optimizer_search_space,
    ga_config=GA_CONFIG,
)

GA_ANALYSIS_RUN_ID = analysis_result["GA_ANALYSIS_RUN_ID"]
GA_ANALYSIS_DIR = analysis_result["GA_ANALYSIS_DIR"]
GA_ANALYSIS_FIG_DIR = analysis_result["GA_ANALYSIS_FIG_DIR"]
GA_ANALYSIS_TABLE_DIR = analysis_result["GA_ANALYSIS_TABLE_DIR"]
GA_ANALYSIS_META_DIR = analysis_result["GA_ANALYSIS_META_DIR"]

In [ ]:
print("\n" + "=" * 80)
print("MULTI-CRITERIA TRADE-OFF ANALYSIS")
print("=" * 80)

# This cell now just summarizes the results produced by workflows.c23_stage_optimizer.

diagnostic_source = ga_ranked_df[ga_ranked_df["status"] == "OK"].copy()
if diagnostic_source.empty:
    diagnostic_source = ga_ranked_df.copy()
    print("Warning: no feasible designs found, using all designs for diagnostic ranking.")

diagnostic_df = analysis_result["diagnostic_df"]
gen_detail = analysis_result["gen_detail"]
final_summary = analysis_result["final_summary"]

print("\nTop designs by different criteria:")
print("\n1. LOWEST EMBODIED CARBON:")
print(analysis_result["cost_winners"].to_string(index=False))
print("\n2. HIGHEST REUSE RATE:")
print(analysis_result["reuse_winners"].to_string(index=False))
print("\n3. LOWEST WASTE VOLUME:")
print(analysis_result["waste_winners"].to_string(index=False))
print("\n4. OVERALL BEST (multi-criteria balanced):")
print(analysis_result["overall_winners"].to_string(index=False))

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Best fitness: {final_summary['best_fitness']:.6f}")
print(f"Best cost: {final_summary['best_cost']:.2f} kg CO2e")
print(f"Best reuse: {final_summary['best_reuse']:.1f}%")
print(f"Best found generation: {final_summary['best_found_generation']}")

# EXPORT

In [ ]:
export_result = stage_optimizer.export_ga_results(
    ga_history_df=ga_history_df,
    ga_ranked_df=ga_ranked_df,
    best_overall=best_overall,
    fixed_norm_constants=FIXED_NORMALIZATION_CONSTANTS,
    ga_config=GA_CONFIG,
    analysis_export_dir=GA_ANALYSIS_DIR,
)

print(f"GA history exported: {export_result['ga_history_path']}")
print(f"GA ranked table exported: {export_result['ga_ranked_path']}")
print(f"Best design exported: {export_result['ga_best_design_path']}")
print(f"Analysis folder exported: {export_result['analysis_export_dir']}")